In [1]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
import pandas as pd
import csv

FILE_NAME = 'mc_questions_file-1.csv'
LOCALE_COL = 'country'

# Tell pandas to ignore quotes entirely
df = pd.read_csv(FILE_NAME, engine='python', on_bad_lines='skip')

# Non-latin (China), Low-Resource (Ethiopia), Additional (Mexico, UK)
locales = ['China', 'Ethiopia', 'Mexico', 'UK']

sampled_dfs = []
for loc in locales:
    loc_df = df[df[LOCALE_COL] == loc]
    # Use random_state=42 for reproducibility, sample 300 rows
    #taking just 300 samples each locale to save the execution time as file is too big
    sampled = loc_df.sample(n=min(300, len(loc_df)), random_state=42)
    sampled_dfs.append(sampled)

mcq_df = pd.concat(sampled_dfs).reset_index(drop=True)
print(f"Total Data rows: {len(mcq_df)}")

Total Data rows: 1200


In [3]:
print(mcq_df.head(5))

           MCQID         ID country  \
0   Na-ko-45_586   Na-ko-45   China   
1  New-ch-79_765  New-ch-79   China   
2  New-as-30_148  New-as-30   China   
3   New-ch-81_53  New-ch-81   China   
4  New-ch-13_179  New-ch-13   China   

                                              prompt  \
0  What is the most popular honeymoon destination...   
1  What leisure activities do retired men typical...   
2  What sport do women in China like to watch? Wi...   
3  What are the typical housewarming gifts in Chi...   
4  What is the most famous alcohol brand in China...   

                                             choices  \
0  {\n    "A": "jeju island",\n    "B": "mashhad"...   
1  {\n    "A": "fix or build things",\n    "B": "...   
2  {\n    "A": "badminton",\n    "B": "baseball",...   
3  {\n    "A": "bicycle",\n    "B": "cash",\n    ...   
4  {\n    "A": "jinro is back",\n    "B": "moutai...   

                                    choice_countries answer_idx  
0  {\n    "A": "South_Kor

In [4]:
!pip install -U bitsandbytes>=0.46.1

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    print("GPU detected — loading with 4-bit quantisation")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    print("No GPU — loading in float32 on CPU (slow, but works)")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,
        device_map=None
    )

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully!")

GPU detected — loading with 4-bit quantisation


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Model loaded successfully!


In [6]:
import json
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score

baseline_predictions = []

def generate_response(prompt):
    messages = [{"role": "user", "content": prompt}]

    # apply chat template to get the formatted string tokenize=false
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # tokenize the text explicitly to get proper input_ids and attention_mask
    model_inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():

        # temperature = 0 as per the assignment question
        generated_ids = model.generate(**model_inputs, max_new_tokens=100, do_sample=False, temperature=0.0)

    # safely get the length from the input_ids tensor specifically
    input_length = model_inputs.input_ids.shape[1]

    # decode the newly generated tokens
    decoded = tokenizer.batch_decode(generated_ids[:, input_length:], skip_special_tokens=True)[0]
    return decoded.strip()

for _, row in tqdm(mcq_df.iterrows(), total=len(mcq_df), desc="Running Baseline Inference"):
    locale = row[LOCALE_COL]
    q = row['prompt']

    # Safely parse choices
    try:
        choices_dict = json.loads(row['choices'])
    except:
        choices_dict = {}

    opt_a = choices_dict.get("A", "")
    opt_b = choices_dict.get("B", "")
    opt_c = choices_dict.get("C", "")
    opt_d = choices_dict.get("D", "")

    opts = f"A) {opt_a}\nB) {opt_b}\nC) {opt_c}\nD) {opt_d}"

    #prompt
    prompt = f"Question: {q}\nOptions:\n{opts}\nWhich option is correct? Reply with only the letter A, B, C, or D."

    ans = generate_response(prompt)

    #parsinng
    cleaned_ans = ''.join([c for c in ans if c.upper() in ['A', 'B', 'C', 'D']])
    pred_letter = cleaned_ans[0].upper() if len(cleaned_ans) > 0 else "N"

    baseline_predictions.append({
        'ID': row.get('id', row.get('MCQID', '')),
        'Locale': locale,
        'Prediction': pred_letter,
        'RawResponse': ans,
        'GroundTruth': row.get('answer_idx', '')
    })

# save results
baseline_df = pd.DataFrame(baseline_predictions)
baseline_df.to_csv("baseline_mcq_prediction.csv", index=False)


index_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D', '0': 'A', '1': 'B', '2': 'C', '3': 'D'}
baseline_df['GroundTruth_Mapped'] = baseline_df['GroundTruth'].map(lambda x: index_to_letter.get(x, str(x).upper()))

baseline_df['GroundTruth_Mapped'] = baseline_df['GroundTruth_Mapped'].astype(str).str.strip().str.upper()
baseline_df['Prediction'] = baseline_df['Prediction'].astype(str).str.strip().str.upper()

print("\n=== BASELINE Evaluation Analysis ===")
overall_acc = accuracy_score(baseline_df['GroundTruth_Mapped'], baseline_df['Prediction'])
print(f"Overall Baseline Accuracy: {overall_acc:.4f}\n")

results = []
locales = baseline_df['Locale'].unique()

for loc in locales:
    loc_df = baseline_df[baseline_df['Locale'] == loc]
    acc = accuracy_score(loc_df['GroundTruth_Mapped'], loc_df['Prediction'])
    results.append({'Locale': loc, 'Accuracy': acc, 'Total Samples': len(loc_df)})

res_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)

if not res_df.empty:
    best_acc = res_df.iloc[0]['Accuracy']
    res_df['Gap_vs_Best'] = best_acc - res_df['Accuracy']

print("Per-Locale Baseline Breakdown:")
print(res_df.to_string(index=False))

Running Baseline Inference:   0%|          | 0/1200 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for ope


=== BASELINE Evaluation Analysis ===
Overall Baseline Accuracy: 0.4567

Per-Locale Baseline Breakdown:
  Locale  Accuracy  Total Samples  Gap_vs_Best
  Mexico  0.490000            300     0.000000
      UK  0.470000            300     0.020000
Ethiopia  0.446667            300     0.043333
   China  0.420000            300     0.070000


In [7]:
import ast

def generate_response(prompt):
    messages = [{"role": "user", "content": prompt}]

    # apply chat template to get the formatted string tokenize=false
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # tokenize the text explicitly to get proper input_ids and attention_mask
    model_inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():

        # temperature = 0 as per the assignment question
        generated_ids = model.generate(**model_inputs, max_new_tokens=100, do_sample=False, temperature=0.0)

    # safely get the length from the input_ids tensor specifically
    input_length = model_inputs.input_ids.shape[1]

    # decode the newly generated tokens
    decoded = tokenizer.batch_decode(generated_ids[:, input_length:], skip_special_tokens=True)[0]
    return decoded.strip()

predictions = []

for _, row in tqdm(mcq_df.iterrows(), total=len(mcq_df), desc="Running Inference"):
    locale = row[LOCALE_COL]

    # question from the dataset
    q = row['prompt']

    # choices available
    raw_choices = row['choices']
    if isinstance(raw_choices, str):
        #string to options
        choices_list = ast.literal_eval(raw_choices)

    else:
        # if pandas already parsed it as a list
        choices_list = raw_choices


    choices_dict = json.loads(row['choices'])

    opt_a = choices_dict.get("A", "")
    opt_b = choices_dict.get("B", "")
    opt_c = choices_dict.get("C", "")
    opt_d = choices_dict.get("D", "")


    opts = f"A) {opt_a}\nB) {opt_b}\nC) {opt_c}\nD) {opt_d}"

    # persona improvement (beyind baseline 1)
    prompt = f"""You are an esteemed cultural anthropologist native to {locale}.
You have deep knowledge of {locale}'s daily life, which often differs significantly from Western norms. Do not rely on Eurocentric defaults.
Question: {q}
Options:
{opts}
Analyze the options based strictly on the local culture of {locale}.
Output your final answer as a valid JSON object with a single key 'answer' and the value as the letter (A, B, C, or D).
Example: {{"answer": "B"}}"""

    ans = generate_response(prompt)

    # JSON parsing (beyond baseline 2)
    try:
        start_idx = ans.find('{')
        end_idx = ans.rfind('}') + 1
        json_str = ans[start_idx:end_idx]
        pred_dict = json.loads(json_str)
        pred_letter = pred_dict.get('answer', 'N').upper()
    except:
        pred_letter = "N" # Fallback if JSON generation fails

    predictions.append({
        'ID': row.get('id', _),
        'Locale': locale,
        'Prediction': pred_letter,
        'GroundTruth': row.get('answer', row.get('Answer', '')),
        'RawResponse': ans
    })

# Save results
results_df = pd.DataFrame(predictions)
results_df.to_csv("improved_mcq_prediction.csv", index=False)
print("file saved as 'improved_mcq_prediction.csv'")

Running Inference:   0%|          | 0/1200 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

file saved as 'improved_mcq_prediction.csv'


In [8]:
correct_answers = mcq_df['answer_idx'].values

# map the indices (0, 1, 2, 3) to letters (A, B, C, D)
index_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D',
                   '0': 'A', '1': 'B', '2': 'C', '3': 'D'}

mapped_answers = [index_to_letter.get(ans, str(ans).upper()) for ans in correct_answers]

results_df['GroundTruth'] = mapped_answers
results_df['GroundTruth'] = results_df['GroundTruth'].astype(str).str.strip().str.upper()
results_df['Prediction'] = results_df['Prediction'].astype(str).str.strip().str.upper()

# save results
results_df.to_csv("predictions_mcq_improved_FIXED.csv", index=False)
print("Saved fixed predictions to 'predictions_mcq_improved_FIXED.csv'\n")

# evaluation Analysis
print("=== Fixed Evaluation Analysis ===")
overall_acc = accuracy_score(results_df['GroundTruth'], results_df['Prediction'])
print(f"Overall Model Accuracy: {overall_acc:.4f}\n")

locales = ['China', 'Ethiopia', 'Mexico', 'UK']
results = []

for loc in locales:
    loc_df = results_df[results_df['Locale'] == loc]
    acc = accuracy_score(loc_df['GroundTruth'], loc_df['Prediction'])
    results.append({'Locale': loc, 'Accuracy': acc, 'Total Samples': len(loc_df)})

res_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)
best_acc = res_df.iloc[0]['Accuracy']
res_df['Gap_vs_Best'] = best_acc - res_df['Accuracy']

print("Per-Locale Breakdown:")
print(res_df.to_string(index=False))

# errors
errors = results_df[results_df['Prediction'] != results_df['GroundTruth']]
errors.sample(min(15, len(errors)), random_state=42).to_csv("error_analysis.csv", index=False)
print("\nSaved 15 accurate errors to 'colab_error_analysis.csv' for your PDF report.")

Saved fixed predictions to 'predictions_mcq_improved_FIXED.csv'

=== Fixed Evaluation Analysis ===
Overall Model Accuracy: 0.7942

Per-Locale Breakdown:
  Locale  Accuracy  Total Samples  Gap_vs_Best
      UK  0.880000            300     0.000000
  Mexico  0.840000            300     0.040000
   China  0.833333            300     0.046667
Ethiopia  0.623333            300     0.256667

Saved 15 accurate errors to 'colab_error_analysis.csv' for your PDF report.


In [9]:
file_names = {
    'China': 'China_questions.csv',
    'Mexico': 'Mexico_questions.csv',
    'UK': 'UK_questions.csv',
    'West_Java': 'West_Java_questions.csv'
}

sampled_dfs = []

for locale, file_path in file_names.items():
    df = pd.read_csv(file_path)
    # add locale column in dataset
    df['Locale'] = locale
    sampled_dfs.append(df)
    print(f'total rows in {locale}: {len(df)}')


# combine into a single dataframe for inference
saq_df = pd.concat(sampled_dfs).reset_index(drop=True)
print(f"total SAQ rows ready for inference: {len(saq_df)}")

total rows in China: 500
total rows in Mexico: 500
total rows in UK: 500
total rows in West_Java: 500
total SAQ rows ready for inference: 2000


In [10]:
def generate_response(prompt):
    messages = [{"role": "user", "content": prompt}]

    # formatted strings
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # tokenization
    model_inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        # temp is 0
        generated_ids = model.generate(**model_inputs, max_new_tokens=50, do_sample=False, temperature=0.0)
    input_length = model_inputs.input_ids.shape[1]

    # decode tokens
    decoded = tokenizer.batch_decode(generated_ids[:, input_length:], skip_special_tokens=True)[0]
    return decoded.strip()

predictions = []

for _, row in tqdm(saq_df.iterrows(), total=len(saq_df), desc="Running SAQ Baseline"):
    locale = row['Locale']
    q = row['Question']

    # baseline prompt to generate ans
    prompt = f"Answer the following question about {locale} briefly.\nQuestion: {q}\nAnswer:"

    # preditions
    ans = generate_response(prompt)

    predictions.append({
        'ID': row.get('ID', ''),
        'Locale': locale,
        'Topic': row.get('Topic', ''),
        'Question': q,
        'Prediction': ans
    })

# save results
saq_results_df = pd.DataFrame(predictions)
saq_results_df.to_csv("colab_predictions_saq_baseline.csv", index=False)
print("Inference complete. File saved as 'colab_predictions_saq_baseline.csv'")

Running SAQ Baseline:   0%|          | 0/2000 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

Inference complete. File saved as 'colab_predictions_saq_baseline.csv'


In [11]:
from google.colab import files

files.download('colab_predictions_saq_baseline.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>